# Tugas 1B: Advanced TF-IDF & Text Summarization (Manchester Football)

**Objective:** Menganalisis teks berita sepak bola menggunakan TF-IDF manual, analisis kata 'manchester', dan Text Summarization.

## 1. Persiapan Data & Library

In [ ]:
import pandas as pd
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.metrics.pairwise import cosine_similarity

# Korpus Manchester
corpus = [
    "Manchester United adalah klub bola besar di Inggris",
    "Manchester City menjuarai liga Inggris musim ini",
    "Derby Manchester selalu dinantikan oleh fans bola"
]

stopwords = set(["adalah", "di", "ini", "oleh", "selalu"])
print("Corpus Manchester loaded.")

## 2. Preprocessing & Tokenization

In [ ]:
def preprocess(text, remove_stop=True):
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = text.split()
    if remove_stop:
        tokens = [t for t in tokens if t not in stopwords]
    return tokens

tokenized_corpus = [preprocess(d) for d in corpus]
vocab = sorted(list(set([word for doc in tokenized_corpus for word in doc])))

print(f"Vocab Size: {len(vocab)}")

## 3. Perhitungan Manual TF-IDF

In [ ]:
def get_tf_matrix(token_docs, vocabulary):
    tf_matrix = []
    for doc in token_docs:
        row = []
        total_terms = len(doc)
        for word in vocabulary:
            count = doc.count(word)
            row.append(count / total_terms if total_terms > 0 else 0)
        tf_matrix.append(row)
    return np.array(tf_matrix)

def get_idf_vector(token_docs, vocabulary):
    N = len(token_docs)
    idf_vector = []
    for word in vocabulary:
        df = sum(1 for doc in token_docs if word in doc)
        idf_vector.append(math.log10(N / df))
    return np.array(idf_vector)

tf_matrix = get_tf_matrix(tokenized_corpus, vocab)
idf_vector = get_idf_vector(tokenized_corpus, vocab)
tfidf_matrix = tf_matrix * idf_vector

df_tfidf = pd.DataFrame(tfidf_matrix, columns=vocab, index=[f"Doc {i+1}" for i in range(len(corpus))])

## 4. Analisis Kata Spesifik: 'manchester'
Sesuai instruksi, kita akan melihat bobot TF-IDF untuk kata **'manchester'**.

In [ ]:
target_word = 'manchester'
if target_word in df_tfidf.columns:
    word_scores = df_tfidf[target_word]
    print(f"Skor TF-IDF untuk kata '{target_word}':")
    print(word_scores)
    
    print(f"\nCatatan: Karena '{target_word}' muncul di seluruh (3) dokumen, nilai IDF-nya adalah log(3/3) = 0.")
    print(f"Oleh karena itu, skor TF-IDF untuk '{target_word}' di semua dokumen adalah 0.")
else:
    print(f"Kata '{target_word}' tidak ditemukan dalam vocab.")

## 5. Implementasi Text Summarization (Sederhana)
Dihitung berdasarkan rata-rata skor TF-IDF setiap kalimat.

In [ ]:
sentence_scores = []
for i in range(len(corpus)):
    scores = tfidf_matrix[i]
    # Hitung rata-rata hanya untuk kata yang punya skor TF-IDF (tidak termasuk yang 0 seperti 'manchester')
    avg_score = np.mean(scores[scores > 0]) if any(scores > 0) else 0
    sentence_scores.append(avg_score)

df_summary = pd.DataFrame({
    'Sentence': corpus,
    'Importance Score': sentence_scores
}).sort_values(by='Importance Score', ascending=False)

print("Hasil Summarization (Rivalitas Manchester):")
display(df_summary)

best_sentence = df_summary.iloc[0]['Sentence']
print(f"\nRingkasan Utama: {best_sentence}")

## 6. Visualisasi Full TF-IDF Heatmap

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(df_tfidf, annot=True, cmap="YlOrRd")
plt.title("Full TF-IDF Heatmap: Manchester Analysis")
plt.show()

## 7. Kesimpulan & Insight
1. **Analisis 'manchester'**: Kata ini muncul di semua dokumen berita olahraga Manchester ini, sehingga nilai IDF-nya 0. Ini berarti kata tersebut tidak membantu membedakan berita satu dengan lainnya.
2. **Summarization**: Teknik ini sangat berguna untuk mencari kalimat 'kunci' dalam sebuah berita olahraga yang panjang.
3. **Regex**: Digunakan untuk membersihkan karakter khusus agar tokenisasi lebih akurat.